# 01. Data Quality Audit & Cleaning
**Author:** Renaldy Bilal Setyawan | **Stack:** DuckDB, JupySQL, Python (Pandas)  
**Dataset:** Gayanara E-Commerce  

---

### 📌 Overview
This notebook executes **Phase 1 (Data Quality Audit)** of the Gayanara analytics pipeline. Before conducting revenue or retention modeling, we validate data integrity, missing values, duplicate records, and category standardization across the 5 core tables.

---

### ⚠️ About this dataset
Gayanara is a **synthetically generated** e-commerce dataset (source: ngulikdata). It is 
used here to demonstrate analytical workflow — data auditing, SQL modeling, segmentation, 
and BI delivery — not to produce real business conclusions.

Where the analysis surfaces artifacts of the generating process rather than genuine 
business signal, this is stated explicitly (see Notebook 02, Task 2 on category 
distribution and Task 7 on retention curve shape). Distinguishing the two is treated 
as part of the analysis.

## 🛠️ Task 0: Environment Setup & Data Ingestion
**Objective:** Initialize the analytical database and ingest raw CSV files into relational tables.

To ensure high-performance querying without needing a standalone database server, this project utilizes **DuckDB**. This allows for incredibly fast, out-of-core SQL aggregations directly within the notebook. 

**The setup process below performs the following:**
1. Establishes a connection to a local `gayanara.db` file.
2. Ingests raw CSV data from the `Dataset` directory directly into 5 core SQL tables using `read_csv_auto`.
3. Binds the DuckDB connection to the `jupysql` engine to enable native `%%sql` execution in subsequent cells.

In [16]:
import duckdb, os

DATA_DIR = 'Dataset/gayanara'

if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(
        f"Could not find '{DATA_DIR}'. Launch Jupyter from the repository root, "
        f"or update DATA_DIR to point at the CSV folder."
    )

con = duckdb.connect('gayanara.db')
print(f"Connected. Reading from {DATA_DIR}/")

Connected. Reading from Dataset/gayanara/


In [17]:
for table in ['customers', 'order_items', 'orders', 'products', 'reviews']:
    con.execute(f"""
        CREATE OR REPLACE TABLE {table} AS
        SELECT * FROM read_csv_auto('{DATA_DIR}/{table}.csv');
    """)

con.execute("SHOW TABLES;").df()

,name
0,customers
1,order_items
2,orders
3,products
4,reviews


## 🔍 Task 1: Null Value Audit across Core Tables
**Business Question:** How complete are critical fields like customer phone numbers, product materials, and review texts?  
**Target Columns:** `customers.phone`, `products.material`, `reviews.review_text`

In [18]:
%load_ext sql
%sql con
%config SqlMagic.displaylimit = 50

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [19]:
%%sql
SHOW TABLES;

Running query in 'DuckDBPyConnection'

name
customers
order_items
orders
products
reviews


`customers.phone`

In [20]:
%%sql
SELECT
  'customers' AS table_name,
  'phone' AS column_name,
  COUNT(*) AS total_rows,
  SUM(
    CASE
      WHEN phone IS NULL THEN 1
      ELSE 0
    END
  ) AS null_count,
  ROUND(
    SUM(
      CASE
        WHEN phone IS NULL THEN 1
        ELSE 0
      END
    ) * 100.0 / COUNT(*),
    2
  ) AS null_percentage
FROM
  customers

Running query in 'DuckDBPyConnection'

table_name,column_name,total_rows,null_count,null_percentage
customers,phone,800,40,5.0


`products.material`

In [21]:
%%sql
SELECT
  'products' AS table_name,
  'material' AS column_name,
  SUM(
    CASE
      WHEN material IS NULL THEN 1
      ELSE 0
    END
  ) AS null_count,
  ROUND(
    SUM(
      CASE
        WHEN material IS NULL THEN 1
        ELSE 0
      END
    ) * 100 / COUNT(*),
    2
  ) AS null_percentage
FROM
  products

Running query in 'DuckDBPyConnection'

table_name,column_name,null_count,null_percentage
products,material,9,3.0


`reviews.review_text`

In [22]:
%%sql
SELECT
  'reviews' AS table_name,
  'review_text' AS column_name,
  SUM(
    CASE
      WHEN review_text IS NULL THEN 1
      ELSE 0
    END
  ) AS null_count,
  ROUND(
    SUM(
      CASE
        WHEN review_text IS NULL THEN 1
        ELSE 0
      END
    ) * 100 / COUNT(*),
    2
  ) AS null_percentage
FROM
  reviews

Running query in 'DuckDBPyConnection'

table_name,column_name,null_count,null_percentage
reviews,review_text,120,8.0


**Target Columns:** `customers.phone`, `products.material`, `reviews.review_text`

In [23]:
%%sql
SELECT
  'customers' AS table_name,
  'phone' AS column_name,
  COUNT(*) AS total_rows,
  SUM(
    CASE
      WHEN phone IS NULL THEN 1
      ELSE 0
    END
  ) AS null_count,
  ROUND(
    SUM(
      CASE
        WHEN phone IS NULL THEN 1
        ELSE 0
      END
    ) * 100.0 / COUNT(*),
    2
  ) AS null_percentage
FROM
  customers
UNION ALL
SELECT
  'products' AS table_name,
  'material' AS column_name,
  COUNT(*) AS total_rows,
  SUM(
    CASE
      WHEN material IS NULL THEN 1
      ELSE 0
    END
  ) AS null_count,
  ROUND(
    SUM(
      CASE
        WHEN material IS NULL THEN 1
        ELSE 0
      END
    ) * 100.0 / COUNT(*),
    2
  ) AS null_percentage
FROM
  products
UNION ALL
SELECT
  'reviews' AS table_name,
  'review_text' AS column_name,
  COUNT(*) AS total_rows,
  SUM(
    CASE
      WHEN review_text IS NULL THEN 1
      ELSE 0
    END
  ) AS null_count,
  ROUND(
    SUM(
      CASE
        WHEN review_text IS NULL THEN 1
        ELSE 0
      END
    ) * 100.0 / COUNT(*),
    2
  ) AS null_percentage
FROM
  reviews;

Running query in 'DuckDBPyConnection'

table_name,column_name,total_rows,null_count,null_percentage
customers,phone,800,40,5.0
products,material,300,9,3.0
reviews,review_text,1500,120,8.0


> 💡 **Key Audit Findings (Nulls):**
> * **Customers (`phone`):** 5.0% missing records (40 rows). This is acceptable for email-led campaigns, but we must use `COALESCE` or filter these out for SMS marketing lists.
> * **Products (`material`):** 3.0% missing values (9 rows). Low impact, but requires a default label like `'Unknown'` during product mix analysis.
> * **Reviews (`review_text`):** 8.0% missing text (120 rows). Normal e-commerce behavior (users frequently leave star ratings without writing a comment). No action needed.

## 🔍 Task 2: Duplicate Customer Records
**Business Question:** Are there users creating multiple accounts with the same email address?
**Target Table:** `customers`

In [24]:
%%sql
SELECT
  email,
  COUNT(customer_id) AS account_count
FROM
  customers
GROUP BY
  email
HAVING
  COUNT(customer_id) > 1
ORDER BY
  account_count DESC;

Running query in 'DuckDBPyConnection'

email,account_count
wulansuryadi729@yahoo.com,2
baguspermata248@gmail.com,2
rinalestari281@outlook.com,2


> 💡 **Key Audit Findings (Duplicates):**
> * Found 3 unique email addresses linked to multiple `customer_id`s (2 accounts each). 
> * **Decision — no deduplication applied:** These 3 duplicate emails represent 6 of 
>   800 customer records (0.75%). Since RFM segmentation is directional rather than 
>   financial reporting, the effect on segment distribution is negligible. Retaining 
>   `customer_id` as the grain preserves the source system's definition of a customer. 
>   Flagged for the data engineering team to resolve upstream.

## 🔍 Task 3: Product Category Standardization
**Business Question:** Are product categories standardized, or is dirty data splitting our revenue groupings?
**Target Table:** `products`

In [25]:
%%sql
SELECT
  category,
  COUNT(product_id) AS total_products
FROM
  products
GROUP BY
  category
ORDER BY
  category ASC;

Running query in 'DuckDBPyConnection'

category,total_products
Accessories,26
Aksesoris,26
Celana,24
Dress,30
Jacket,29
Jaket,27
Kaos,19
Kemeja,17
Pants,28
Shirt,16


> 💡 **Key Audit Findings (Category Inconsistencies):**
> * The `category` column contains 14 distinct values due to a mix of English and Indonesian terminology (e.g., *Aksesoris* vs *Accessories*) AND capitalization inconsistencies (e.g., *Dress* vs *dress*). 
> * **Action Required:** We must standardize these into 6 clean categories using a `CASE WHEN` mapping before calculating revenue metrics.

In [26]:
%%sql
SELECT
  CASE
    WHEN LOWER(category) IN ('aksesoris', 'accessories') THEN 'Accessories'
    WHEN LOWER(category) IN ('celana', 'pants') THEN 'Pants'
    WHEN LOWER(category) IN ('jaket', 'jacket') THEN 'Jacket'
    WHEN LOWER(category) IN ('kemeja', 'shirt') THEN 'Shirt'
    WHEN LOWER(category) IN ('kaos', 't-shirt', 'tshirt') THEN 'T-Shirt'
    WHEN LOWER(category) IN ('dress') THEN 'Dress'
    ELSE 'Other'
  END AS category_clean,
  COUNT(product_id) AS total_products
FROM
  products
GROUP BY
  category_clean
ORDER BY
  total_products DESC;

Running query in 'DuckDBPyConnection'

category_clean,total_products
Jacket,56
Accessories,52
Pants,52
Shirt,48
Dress,48
T-Shirt,44


In [27]:
%%sql
-- 1. Adding new column to store the clean category
ALTER TABLE products
ADD COLUMN category_clean VARCHAR;

-- 2. Update the new column
UPDATE products
SET
  category_clean = CASE
    WHEN LOWER(category) IN ('aksesoris', 'accessories') THEN 'Accessories'
    WHEN LOWER(category) IN ('celana', 'pants') THEN 'Pants'
    WHEN LOWER(category) IN ('jaket', 'jacket') THEN 'Jacket'
    WHEN LOWER(category) IN ('kemeja', 'shirt') THEN 'Shirt'
    WHEN LOWER(category) IN ('kaos', 't-shirt', 'tshirt') THEN 'T-Shirt'
    WHEN LOWER(category) IN ('dress') THEN 'Dress'
    ELSE 'Other'
  END;

Running query in 'DuckDBPyConnection'

Count
300


In [28]:
%%sql
-- 3. Verify 
SELECT
  category_clean,
  COUNT(*) AS COUNT
FROM
  products
GROUP BY
  category_clean;

Running query in 'DuckDBPyConnection'

category_clean,COUNT
Shirt,48
Dress,48
Accessories,52
Jacket,56
Pants,52
T-Shirt,44


## 🔍 Task 4: Data Integrity (Orphaned Records)
**Business Question:** Are there any items sold in the `order_items` table that do not have a matching record in the `products` table?
**Target Tables:** `order_items` (LEFT JOIN) `products`

In [29]:
%%sql
SELECT
  COUNT(*) AS total_items_sold,
  SUM(
    CASE
      WHEN p.product_id IS NULL THEN 1
      ELSE 0
    END
  ) AS orphaned_items,
  ROUND(
    SUM(
      CASE
        WHEN p.product_id IS NULL THEN 1
        ELSE 0
      END
    ) * 100.0 / COUNT(*),
    2
  ) AS orphaned_percentage
FROM
  order_items oi
  LEFT JOIN products p ON oi.product_id = p.product_id;

Running query in 'DuckDBPyConnection'

total_items_sold,orphaned_items,orphaned_percentage
4986,0,0.0


Closing DuckDBPyConnection

> 💡 **Key Audit Findings (Data Integrity):**
> * **Orphaned Records:** 0% (0 out of 4,986 items sold).
> * **Conclusion:** Excellent referential integrity between `order_items` and `products`. We can safely perform `INNER JOIN` operations for revenue and product-level calculations without dropping any sales data.

---
**✅ Phase 1 (Data Quality Audit) Complete.** The foundation is clean, standardized, and verified.